# Kinase LID Redesign

**BioPipelines example** — de novo backbone design of the adenylate kinase LID domain using RFdiffusion, followed by ProteinMPNN sequence design and AlphaFold2 refolding. Candidates are filtered by RMSD of the fixed regions and pLDDT.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e .
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f environments/biopipelines.yaml -y

Cloning into 'biopipelines'...
remote: Enumerating objects: 7678, done.
remote: Counting objects: 100% (1048/1048), done.
remote: Compressing objects: 100% (390/390), done.
remote: Total 7678 (delta 717), reused 877 (delta 645), pack-reused 6630 (from 1)
Receiving objects: 100% (7678/7678), 16.91 MiB | 27.97 MiB/s, done.
Resolving deltas: 100% (5872/5872), done.
/content/biopipelines
Obtaining file:///content/biopipelines
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 12.8 MB/s eta 0:00:00
  Building editable for biopipelines (pyproject.toml) ... done
  Created wheel for biopipelines: filename=biopipelines-1.1.1-0.ed

In [2]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.pymol import PyMOL

with Pipeline(project="Setup", job="InstallTools"):
    RFdiffusion.install()
    ProteinMPNN.install()
    AlphaFold.install()
    PyMOL.install()

Streaming output truncated to the last 5000 lines.
271150K .......... .......... .......... .......... .......... 57%  483M 6s
271200K .......... .......... .......... .......... .......... 57%  394M 6s
271250K .......... .......... .......... .......... .......... 57%  377M 6s
271300K .......... .......... .......... .......... .......... 57%  761M 6s
271350K .......... .......... .......... .......... .......... 57%  348M 6s
271400K .......... .......... .......... .......... .......... 57%  358M 6s
271450K .......... .......... .......... .......... .......... 57%  429M 6s
271500K .......... .......... .......... .......... .......... 57%  323M 6s
271550K .......... .......... .......... .......... .......... 57%  448M 6s
271600K .......... .......... .......... .......... .......... 57%  241M 6s
271650K .......... .......... .......... .......... .......... 57%  509M 6s
271700K .......... .......... .......... .......... .......... 57%  698M 6s
271750K .......... .......... .......

## Cell 3: Kinase LID Redesign Pipeline

Takes adenylate kinase (PDB: 4AKE) and redesigns the LID domain (residues 118–160, replaced with a new 50–70 residue loop).
The top 3 designs with RMSD < 1.5 Å on the fixed scaffold and highest pLDDT are selected.

In [3]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.conformational_change import ConformationalChange
from biopipelines.panda import Panda
from biopipelines.pymol import PyMOL

with Pipeline(project="AdenylateKinase", job="LID_Redesign"):
    kinase = PDB("4AKE")
    backbones = RFdiffusion(pdb=kinase,
                            contigs='A1-117/50-70/A161-214',
                            num_designs=10)
    sequences = ProteinMPNN(structures=backbones,
                            num_sequences=2,
                            redesigned=backbones.tables.structures.designed)
    refolded = AlphaFold(proteins=sequences)
    conf_change = ConformationalChange(reference_structures=kinase,
                                       target_structures=refolded,
                                       selection=backbones.tables.structures.fixed)
    top3 = Panda(tables=[refolded.tables.confidence,
                          conf_change.tables.changes],
                 operations=[Panda.merge(),
                              Panda.filter("RMSD < 1.5"),
                              Panda.sort("plddt", ascending=False),
                              Panda.head(3)],
                 pool=refolded)
top3

  4AKE not found locally, will download from RCSB

Running PDB (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
Fetching 1 structures
Convert: keep as-is (pdb|cif)
PDB IDs: 4AKE
Custom IDs: 4AKE
Priority: pdbs/ -> RCSB download
Output folder: /content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/001_PDB
Fetching 1 structures (keep as-is (pdb|cif))
Priority: pdbs/ -> RCSB download
Water molecules will be removed

[1/1] Processing 4AKE -> 4AKE
4AKE not found locally, downloading from RCSB
Cached to pdbs/ folder: /content/biopipelines/pdbs/4AKE.pdb
Successfully downloaded 4AKE as 4AKE: 297432 bytes (rcsb_download (PDB))

Successful fetches saved: /content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/001_PDB/structures/structures.csv (1 structures)
Sequences saved: /content/biopipelines/o

StandardizedOutput({'structures': DataStream(name='structures', format='pdb', items=3, files=3, map_table=set), 'msas': DataStream(name='msas', format='a3m', items=3, files=3, map_table=set), 'rendering_parameters': {'structures': {'color_by': 'plddt', 'plddt_upper': 100}}, 'tables': {'result': TableInfo(name='result', path='/content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/006_Panda/tables/merge_filter_sort.csv', columns=['id', 'structure', 'plddt', 'max_pae', 'ptm'], count=variable), 'missing': TableInfo(name='missing', path='/content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/006_Panda/tables/missing.csv', columns=['id', 'removed_by', 'cause'], count=variable), 'structures': TableInfo(name='structures', path='/content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/006_Panda/structures/structures_map.csv', columns=['id', 'file'], count=1), 'confidence': TableInfo(name='confidence', path='/content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/006_Panda/tables/confidence.csv', columns=['id', 'structure', 'plddt', 'max_pae', 'ptm'], count=1), 'msas': TableInfo(name='msas', path='/content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/006_Panda/msas/msas_map.csv', columns=['id', 'sequences.id', 'sequence', 'msa_file'], count=1)}, 'output_folder': '/content/biopipelines/outputs/AdenylateKinase/LID_Redesign_001/006_Panda'})